In [15]:
import pickle

import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error

In [2]:
from sklearn.pipeline import make_pipeline

In [3]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("green-taxi-duration")

2026/09/19 14:45:34 INFO mlflow.tracking.fluent: Experiment with name 'green-taxi-duration' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/mlops-zoomcamp/04-deployment/web-service-mlflow/artifacts/1', creation_time=1789829134467, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789829134467, lifecycle_stage='active', name='green-taxi-duration', tags={}, trace_location=None, workspace='default'>

In [4]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [12]:
df_train = read_dataframe('data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('data/green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [ ]:
with mlflow.start_run()as run:
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    # rmse = mean_squared_error(y_pred, y_val, squared=False) --> scikilearn 1.6
    rmse = root_mean_squared_error(y_val, y_pred)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)

    # mlflow.sklearn.log_model(pipeline, artifact_path="model")  # → mlflow2.X
    model_info = mlflow.sklearn.log_model(pipeline, name="model") # → mlflow3.X
    print(model_info.model_uri)   # → models:/m-xxxxxxxx

 
RUN_ID = run.info.run_id
print("run_id   :", RUN_ID)
print("model_uri:", model_info.model_uri)

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 6.7558229919200725
models:/m-5e276730f7004842b7c3a36ea11b5044
🏃 View run smiling-mouse-209 at: http://127.0.0.1:5000/#/experiments/1/runs/0ee6291ee44b48b4a7aae38f11a01465
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [17]:
print(model_info.model_uri)   # → models:/m-xxxxxxxx

models:/m-5e276730f7004842b7c3a36ea11b5044


In [18]:
from mlflow.tracking import MlflowClient


In [21]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
RUN_ID = '0ee6291ee44b48b4a7aae38f11a01465'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [28]:
#path = client.download_artifacts(run_id=RUN_ID, path='dict_vectorizer.bin') -->mlflow 2.X

In [ ]:
#code mlflwo3.0 compatible pour voir le contenu du model
for a in mlflow.artifacts.list_artifacts(artifact_uri=model_info.model_uri):
    print(a.path)

MLmodel
conda.yaml
model.skops
python_env.yaml
requirements.txt


In [ ]:
# plus besoin de ces lignes grâce au pipeline
#with open(path, 'rb') as f_out:
#    dv = pickle.load(f_out)

NameError: name 'path' is not defined

In [ ]:
#dv

DictVectorizer()

In [31]:
#à titre pédagogique, pour quand m^me voir le le DictVectoizer, à partir du pipeline
pipeline = mlflow.sklearn.load_model(model_info.model_uri)
print(pipeline.named_steps)
dv = pipeline.named_steps['dictvectorizer']
print(len(dv.feature_names_))

{'dictvectorizer': DictVectorizer(), 'randomforestregressor': RandomForestRegressor(max_depth=20, min_samples_leaf=10, n_jobs=-1,
                      random_state=0)}
13221
